In [1]:
import pandas as pd
sheet_names =['2015','2016','2017','2018']
sheet_datas = pd.read_excel('C:/Develop/深圳42/data/sales.xlsx',sheet_name=sheet_names)
# 这里加载excel 里面有多个sheet 
# 我们把工作表的名字 放到列表中, 传给 sheet_name参数, 返回一个字典, key工作表的名字, value就是对应数据的dataframe对象

## 数据清洗

In [5]:
#查看数据的基本情况
for sheet_name in sheet_names:
    print(sheet_datas[sheet_name].head())
    print(sheet_datas[sheet_name].info())
    print(sheet_datas[sheet_name].describe())
    print('==============='+sheet_name+'===============')

          会员ID         订单号       提交日期    订单金额
0  15278002468  3000304681 2015-01-01   499.0
1  39236378972  3000305791 2015-01-01  2588.0
2  38722039578  3000641787 2015-01-01   498.0
3  11049640063  3000798913 2015-01-01  1572.0
4  35038752292  3000821546 2015-01-01    10.1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30774 entries, 0 to 30773
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   会员ID    30774 non-null  int64         
 1   订单号     30774 non-null  int64         
 2   提交日期    30774 non-null  datetime64[ns]
 3   订单金额    30774 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 961.8 KB
None
               会员ID           订单号           订单金额
count  3.077400e+04  3.077400e+04   30774.000000
mean   2.918779e+10  4.020414e+09     960.991161
std    1.385333e+10  2.630510e+08    2068.107231
min    2.670000e+02  3.000305e+09       0.500000
25%    1.944122e+10  3.885510e+

In [6]:
# 有两年的数据金额有1条缺失  订单金额差异过大, 小的几毛钱,还有0.0的 最大的十几万
data_merge = pd.concat([sheet_datas[i] for i in sheet_names])

In [8]:
# 去掉缺失值
data_merge.dropna(inplace=True)
data_merge.shape[0]

204238

In [9]:
# 保留订单金额>1
data_merge.query('订单金额>1',inplace=True)

In [12]:
# 分年度来计算RFM
data_merge['年份'] = data_merge['提交日期'].dt.year

In [16]:
data_merge.groupby('年份')['提交日期'].agg('max')

年份
2015   2015-12-31
2016   2016-12-31
2017   2017-12-31
2018   2018-12-31
Name: 提交日期, dtype: datetime64[ns]

In [18]:
data_merge['计算日期']= data_merge.groupby('年份')['提交日期'].transform('max')
# 分组变换, 类似与SQL的partition by 分组计算后, 数据的量不会改变, 每组的数据会算出一个相同的值来

In [19]:
data_merge

,会员ID,订单号,提交日期,订单金额,年份,计算日期
0,15278002468,3000304681,2015-01-01,499.0,2015,2015-12-31
1,39236378972,3000305791,2015-01-01,2588.0,2015,2015-12-31
2,38722039578,3000641787,2015-01-01,498.0,2015,2015-12-31
3,11049640063,3000798913,2015-01-01,1572.0,2015,2015-12-31
4,35038752292,3000821546,2015-01-01,10.1,2015,2015-12-31
...,...,...,...,...,...,...
81344,39229485704,4354225182,2018-12-31,249.0,2018,2018-12-31
81345,39229021075,4354225188,2018-12-31,89.0,2018,2018-12-31
81346,39288976750,4354230034,2018-12-31,48.5,2018,2018-12-31
81347,26772630,4354230163,2018-12-31,3196.0,2018,2018-12-31


## 计算RFM的聚合值

In [21]:
data_merge['购买日期差值'] = (data_merge['计算日期']-data_merge['提交日期']).dt.days

In [24]:
rfm_result = data_merge.groupby(['年份','会员ID']).agg({'购买日期差值':'min','订单号':'count','订单金额':'sum'}).reset_index()

In [27]:
rfm_result.columns=['年份', '会员ID', 'R', 'F', 'M']

## RFM 三个维度打分 打3分 3,2,1

In [30]:
rfm_result.iloc[:,2:].describe()

,R,F,M
count,148591.000000,148591.000000,148591.000000
mean,165.524043,1.365002,1323.741329
std,101.988472,2.626953,3753.906883
min,0.000000,1.000000,1.500000
25%,79.000000,1.000000,69.000000
50%,156.000000,1.000000,189.000000
75%,255.000000,1.000000,1199.000000
max,365.000000,130.000000,206251.800000


In [34]:
# f 从分位数观察发现, 大多数用户一年以内购买的次数都是1次 2次以上的频率就不算低了, 5次以上的算是高频用户
# r 和 M  这里考虑使用1/4 3/4 分位数作为阈值
r_bins = [-1,79,255,365]
f_bins = [0,2,5,130] # (]
m_bins = [0,69,1199,206252]

In [35]:
# labels=[1,2,3] 这种写法, 写死了叫硬编码, 如果有办法替换, 尽量避免
# range(开始的值, 结束的值, 步长) 步长默认是1
# range(len(r_bins)-1,0,-1)  从len(r_bins)-1开始  到0结束 每次-1   结束的值不会包含进来
rfm_result['r_score']=pd.cut(rfm_result['R'],bins=r_bins,labels=[i for i in range(len(r_bins)-1,0,-1)])
rfm_result['f_score'] =pd.cut(rfm_result['F'],bins=f_bins,labels=[i for i in range(1,len(f_bins))])
rfm_result['m_score'] = pd.cut(rfm_result['M'],bins=m_bins,labels=[i for i in range(1,len(m_bins))])

In [ ]:
##验证数据是否正确

In [36]:
rfm_result.shape[0]

148591

In [39]:
rfm_result['r_score'].value_counts()

2    74059
3    37603
1    36929
Name: r_score, dtype: int64

In [42]:
rfm_result['f_score'].value_counts()

1    142001
2      4534
3      2056
Name: f_score, dtype: int64

In [44]:
rfm_result['m_score'].value_counts()

148591

In [50]:
[i for i in range(len(r_bins)-1,0,-1)]

[3, 2, 1]

In [51]:
[i for i in range(1,len(f_bins))]

[1, 2, 3]

## r/f/m的分数拼接到一起

In [54]:
rfm_result['rfm_label'] = rfm_result['r_score'].astype(str)+rfm_result['f_score'].astype(str)+rfm_result['m_score'].astype(str)

In [58]:
rfm_result.groupby(['rfm_label','年份'])['会员ID'].agg( 'count')

rfm_label  年份  
111        2015    2180
           2016    1498
           2017    3169
           2018    2271
112        2015    3811
                   ... 
332        2018      24
333        2015      15
           2016      28
           2017      87
           2018     355
Name: 会员ID, Length: 88, dtype: int64

In [62]:
rfm_result[(rfm_result['年份']==2015) &(rfm_result['f_score']==2)]

,年份,会员ID,R,F,M,r_score,f_score,m_score,rfm_label
4,2015,525,37,3,213.0,3,2,2,322
33,2015,8358,10,3,2195.9,3,2,3,323
98,2015,41917,232,3,10.0,2,2,1,221
104,2015,45842,54,4,58.9,3,2,1,321
110,2015,48996,125,3,4840.8,2,2,3,223
...,...,...,...,...,...,...,...,...,...
26236,2015,39460064727,363,3,1796.0,1,2,3,123
26647,2015,39478152696,362,3,3057.7,1,2,3,123
26847,2015,39480178314,356,3,5447.0,1,2,3,123
27045,2015,39482874224,362,3,345.0,1,2,2,122


## 数据可视化

In [65]:
display_data = rfm_result.groupby(['rfm_label','年份'],as_index=False)['会员ID'].agg('count')
display_data.columns=['rfm_label','year','number']

In [70]:
from pyecharts.charts import Bar3D
import pyecharts.options as opts
# 颜色池
range_color = ['#313695', '#4575b4', '#74add1', '#abd9e9', '#e0f3f8', '#ffffbf', '#fee090', '#fdae61', '#f46d43', 
'#d73027', '#a50026']
range_max = int(display_data['number'].max())
c = (Bar3D()#设置了一个3D柱形图对象
    .add("",#图例
        [d.tolist() for d in display_data.values],#数据
    xaxis3d_opts=opts.Axis3DOpts(type_="category", name='分组名称'),#x轴数据类型，名称，rfm_group
    yaxis3d_opts=opts.Axis3DOpts(type_="category", name='年份'),#y轴数据类型，名称，year
    zaxis3d_opts=opts.Axis3DOpts(type_="value",    name='会员数量'),#z轴数据类型，名称，number
    ).set_global_opts( # 全局设置
        visualmap_opts=opts.VisualMapOpts(max_=range_max, range_color=range_color), #设置颜色，及不同取值对应的颜色
        title_opts=opts.TitleOpts(title="RFM分组结果"),#设置标题
    )
)
c.render() #在notebook中显示

'C:\\Develop\\深圳42\\day06\\02_代码\\render.html'

In [ ]:
pip install pyecharts==2.0.3